# Project: Interactive Dashboards with Plotly

Create interactive visualizations using Plotly Express: scatter plots, line charts, bar charts, treemaps, and parallel coordinates.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Plotly version:', px.__version__)

In [ ]:
np.random.seed(42)
n = 500
categories = ['Electronics','Clothing','Food','Books','Sports']
regions = ['North','South','East','West']
df = pd.DataFrame({
    'category': np.random.choice(categories, n),
    'region': np.random.choice(regions, n),
    'sales': np.random.lognormal(4, 0.8, n),
    'quantity': np.random.randint(1, 50, n),
    'rating': np.random.uniform(1, 5, n),
    'profit_margin': np.random.uniform(0.05, 0.4, n),
    'month': np.random.randint(1, 13, n),
})
df['revenue'] = df['sales'] * df['quantity']
df['profit'] = df['revenue'] * df['profit_margin']
df.head()

## 1. Interactive Scatter Matrix

In [ ]:
fig = px.scatter_matrix(
    df, dimensions=['sales','quantity','revenue','profit_margin','rating'],
    color='category', title='Scatter Matrix', opacity=0.6)
fig.update_traces(diagonal_visible=False)
fig.show()

## 2. Treemap (Hierarchical View)

In [ ]:
df_agg = df.groupby(['region','category'])[['revenue','profit']].sum().reset_index()
fig = px.treemap(df_agg, path=['region','category'], values='revenue',
                 color='profit', color_continuous_scale='RdYlGn',
                 title='Revenue by Region and Category')
fig.show()

## 3. Parallel Coordinates

In [ ]:
fig = px.parallel_coordinates(
    df, dimensions=['sales','quantity','revenue','profit_margin','rating'],
    color='profit', color_continuous_scale=px.colors.diverging.Tealrose,
    title='Multi-Dimensional View')
fig.show()

## 4. Multi-Chart Dashboard

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Revenue by Category','Sales Distribution',
                    'Profit Margin by Region','Monthly Trend'),
    specs=[[{'type':'bar'},{'type':'box'}],[{'type':'bar'},{'type':'scatter'}]])

cat_rev = df.groupby('category')['revenue'].sum().reset_index()
fig.add_trace(go.Bar(x=cat_rev['category'], y=cat_rev['revenue']), row=1, col=1)
fig.add_trace(go.Box(y=df['sales'], x=df['category']), row=1, col=2)
reg_pm = df.groupby('region')['profit_margin'].mean().reset_index()
fig.add_trace(go.Bar(x=reg_pm['region'], y=reg_pm['profit_margin']), row=2, col=1)
monthly = df.groupby('month')['revenue'].sum().reset_index()
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['revenue'], mode='lines+markers'), row=2, col=2)
fig.update_layout(height=700, showlegend=False, title_text='Multi-Chart Dashboard')
fig.show()

## Summary

- Interactive scatter matrix
- Treemap for hierarchical data
- Parallel coordinates
- Multi-chart dashboard with subplots